In [1]:
import numpy as np
import pandas as pd

In [2]:
np.random.seed(42)
n = 1000

In [3]:
data = pd.DataFrame({
    "age": np.random.randint(18, 65, n),
    "gender": np.random.choice(["男", "女"], n),
    "browse_duration": np.round(np.random.exponential(30, n), 1),  # 分钟
    "add_to_cart": np.random.poisson(3, n),
    "history_purchases": np.random.randint(0, 50, n),
})

data

,age,gender,browse_duration,add_to_cart,history_purchases
0,56,男,34.5,4,20
1,46,女,17.7,1,9
2,32,女,9.6,4,42
3,60,女,175.5,3,14
4,25,男,16.7,1,18
...,...,...,...,...,...
995,22,男,3.5,5,19
996,40,男,16.5,4,9
997,27,女,1.3,4,22
998,61,男,40.4,1,40


In [4]:
score = (
    0.03 * data["age"]
    + 0.5 * (data["gender"] == "女").astype(int)
    + 0.05 * data["browse_duration"]
    + 0.3 * data["add_to_cart"]
    + 0.1 * data["history_purchases"]
    - 4
)

In [5]:
score

0      2.605
1     -0.035
2      3.340
3      9.375
4     -0.315
       ...  
995    0.235
996    0.125
997    0.775
998    4.150
999    3.625
Length: 1000, dtype: float64

In [6]:
prob = 1 / (1 + np.exp(-score))

In [7]:
data["purchased"] = (np.random.random(n) < prob).astype(int)

In [8]:
data

,age,gender,browse_duration,add_to_cart,history_purchases,purchased
0,56,男,34.5,4,20,1
1,46,女,17.7,1,9,0
2,32,女,9.6,4,42,1
3,60,女,175.5,3,14,1
4,25,男,16.7,1,18,1
...,...,...,...,...,...,...
995,22,男,3.5,5,19,0
996,40,男,16.5,4,9,1
997,27,女,1.3,4,22,0
998,61,男,40.4,1,40,1


In [9]:
for col in ["age", "browse_duration", "add_to_cart"]:
    mask = np.random.random(n) < 0.05
    data.loc[mask, col] = np.nan

In [10]:
data.shape

(1000, 6)

In [11]:
data.head()

,age,gender,browse_duration,add_to_cart,history_purchases,purchased
0,56.0,男,34.5,4.0,20,1
1,NaN,女,17.7,1.0,9,0
2,32.0,女,9.6,NaN,42,1
3,60.0,女,175.5,3.0,14,1
4,25.0,男,16.7,1.0,18,1


In [12]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                947 non-null    float64
 1   gender             1000 non-null   object 
 2   browse_duration    949 non-null    float64
 3   add_to_cart        946 non-null    float64
 4   history_purchases  1000 non-null   int32  
 5   purchased          1000 non-null   int64  
dtypes: float64(3), int32(1), int64(1), object(1)
memory usage: 43.1+ KB


In [13]:
data.describe()

,age,browse_duration,add_to_cart,history_purchases,purchased
count,947.000000,949.000000,946.000000,1000.00000,1000.000000
mean,40.882788,29.711486,2.951374,24.43200,0.803000
std,13.540870,29.508555,1.675141,14.24508,0.397931
min,18.000000,0.100000,0.000000,0.00000,0.000000
25%,29.000000,8.400000,2.000000,12.00000,1.000000
50%,41.000000,21.100000,3.000000,24.00000,1.000000
75%,52.000000,40.100000,4.000000,37.00000,1.000000
max,64.000000,192.200000,9.000000,49.00000,1.000000


In [14]:
missing = data.isnull().sum()

In [15]:
missing

age                  53
gender                0
browse_duration      51
add_to_cart          54
history_purchases     0
purchased             0
dtype: int64

In [16]:
missing_pct = (missing / len(data) * 100).round(2)

In [17]:
print(pd.DataFrame({"缺失数": missing, "缺失比例%": missing_pct}))

                   缺失数  缺失比例%
age                 53    5.3
gender               0    0.0
browse_duration     51    5.1
add_to_cart         54    5.4
history_purchases    0    0.0
purchased            0    0.0


In [18]:
for col in ["age", "browse_duration", "add_to_cart"]:
    median_val = data[col].median()
    data[col] = data[col].fillna(median_val)
    print(f"{col}: 用中位数 {median_val} 填充")

print(f"\n剩余缺失值: {data.isnull().sum().sum()}")

age: 用中位数 41.0 填充
browse_duration: 用中位数 21.1 填充
add_to_cart: 用中位数 3.0 填充

剩余缺失值: 0


In [19]:
print("目标变量分布:")
print(data["purchased"].value_counts(normalize=True).round(3))

目标变量分布:
purchased
1    0.803
0    0.197
Name: proportion, dtype: float64


In [20]:
print("\n性别分布:")
print(data["gender"].value_counts())


性别分布:
gender
男    526
女    474
Name: count, dtype: int64


In [21]:
print("\n各特征按购买/未购买分组均值:")
print(data.groupby("purchased").mean(numeric_only=True).round(2))


各特征按购买/未购买分组均值:
             age  browse_duration  add_to_cart  history_purchases
purchased                                                        
0          35.89            16.36         2.50              12.88
1          42.11            32.44         3.07              27.27
